In [2]:
import os
import requests

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=10,
    ).json()

    if not geo.get("results"):
        return f"Could not find city: {city}"

    place = geo["results"][0]
    wx = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,wind_speed_10m",
        },
        timeout=10,
    ).json()

    cur = wx["current"]
    return f"{place['name']}, {place.get('country', '')}: {cur['temperature_2m']}°C, wind {cur['wind_speed_10m']} km/h"

agent = create_agent(llm, tools=[get_weather])

result = agent.invoke(
    {"messages": [{"role":"user", "content":"Is it warmer in paris or Delhi right now ?"}]},
    config={"recursion_limit": 10}
)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Is it warmer in paris or Delhi right now ?
================================== Ai Message ==================================

I'll check the current weather in both Paris and Delhi for you.
Tool Calls:
  get_weather (toolu_01YYbJ6MRQWGpmAW2FB5aUuS)
 Call ID: toolu_01YYbJ6MRQWGpmAW2FB5aUuS
  Args:
    city: Paris
  get_weather (toolu_01T82KSS1oWzf3d9onzzE9Yk)
 Call ID: toolu_01T82KSS1oWzf3d9onzzE9Yk
  Args:
    city: Delhi
================================= Tool Message =================================
Name: get_weather

Paris, France: 12.7°C, wind 5.2 km/h
================================= Tool Message =================================
Name: get_weather

Delhi, India: 27.0°C, wind 5.6 km/h
================================== Ai Message ==================================

Delhi is warmer right now! 

- **Delhi**: 27.0°C
- **Paris**: 12.7°C

Delhi is about 14.3°C warmer than Paris at the moment.


In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

llm_with_tools = llm.bind_tools([get_weather])

def call_model(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode([get_weather]))

builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")
graph = builder.compile()

result = graph.invoke(
    {"messages":[{"role":"user", "content":"Is it warmer in paris or Delhi right now"}]},
    config={"recursion_limit":10}
)

for m in result["messages"]:
    m.pretty_print()

print(graph.get_graph().draw_mermaid())